# Run the experiment on Colab

Runs our programs on the model exactly as described in `EXPERIMENT.md`.

**Before you start**
1. On GitHub, click **Code → Download ZIP** for the `misleading-comments` repo.
2. In Colab: **Runtime → Change runtime type → T4 GPU**.

Then run the cells in order (**Runtime → Run all** works too). Cell 2 asks you to upload the ZIP.

## 1. Check there's a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Upload the repo ZIP

In [ ]:
import glob, os, sys, zipfile
from google.colab import files

uploaded = files.upload()  # pick the ZIP you downloaded from GitHub
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall("/content/repo")
REPO = os.path.dirname(glob.glob("/content/repo/**/run.py", recursive=True)[0])
os.chdir(REPO)
sys.path.insert(0, REPO)
print("Repo ready at", REPO)

## 3. Install and start Ollama

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version")
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama didn't start; see /content/ollama.log")

## 4. Download the model
Uses the exact model name from `run.py` (about 4.7 GB, a few minutes).

In [ ]:
import run
print("Model:", run.MODEL)
!ollama pull {run.MODEL}

## 5. Run
Leave `PROGRAMS` empty to run every program, or list folder names to run only those.

In [ ]:
PROGRAMS = []  # e.g. ["chunk_sums", "longest_within_budget", "count_long_runs"]

names = PROGRAMS or sorted(p for p in os.listdir("programs") if os.path.isdir(os.path.join("programs", p)))
args = " ".join("programs/" + n for n in names)
!python3 run.py {args}
!ollama ps

`ollama ps` above should say **100% GPU**. If it shows CPU, the runtime has no GPU; fix that in step 2 of "Before you start" and rerun.

## 6. Summary

In [ ]:
import json
for path in sorted(glob.glob("results/*/*.json")):
    for r in json.load(open(path)):
        print(f'{r["program"]:<26} prompt {r["prompt"]}  {r["version"]:<15} {"PASS" if r["success"] else "FAIL"}')

## 7. Download the results

In [ ]:
import shutil
shutil.make_archive("/content/results", "zip", REPO, "results")
files.download("/content/results.zip")

Unzip `results.zip`, then add its `results/` files to the repo:
- with git: copy them into your local `results/` folder, commit and push, or
- on GitHub: open the repo, **Add file → Upload files**, and drag in the `results` folder.

Don't edit the result files by hand.